<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/BERT_DistilBert_Twitter_XLM_Robert_(EN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers

In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

In [7]:
df = pd.read_csv('/content/StereoQueerEval_EN_training.tsv',sep='\t')
df.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


In [8]:
df["combined_text"] = (
    "Comment: " + df["yt_comment"].fillna("").astype(str)
    + " Title: " + df["yt_title"].fillna("").astype(str)
    + " Description: " + df["yt_description"].fillna("").astype(str)
)

In [9]:
df.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target,combined_text
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l,Comment: She's living in unrepentant defiance ...
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t,Comment: Rep Mace: Keep on being dedicated to ...
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g","Comment: We all need Jesus, He said no homosex..."
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l,Comment: So can two black men adopt a little w...
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g,Comment: Of course west and US will hate him T...


In [10]:
df['combined_text'][1]

'Comment: Rep Mace: Keep on being dedicated to truth and NOT politically correct propaganda! Thank you for your leadership. Title: GOP Congresswoman uses anti-trans slur in House hearing Description: Rep. Nancy Mace (R-SC) was challenged by Rep. Gerry Connolly (D-VA) for her use of anti-transgender slur while asking about USAID funding at a House Oversight Committee hearing. She responded with “I don’t really care.” Mace called herself “pro-transgender rights” as recently as 2023. Since then she has repeatedly used anti-transgender rhetoric. #[CHANNEL] #news'

In [13]:
df["stereotype"].value_counts(normalize=True)

,proportion
stereotype,
no,0.556373
yes,0.443627


In [11]:
X = list(df['combined_text'])
y = df['stereotype'].map({'no':0, 'yes':1}).tolist()

In [12]:
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=45)

### BERT

In [14]:
from transformers import  AutoTokenizer

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [19]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [20]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [21]:
from transformers import AutoModelForSequenceClassification,Trainer,TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [23]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,1.110186
20,0.778267
30,0.741691
40,0.702865
50,0.732158
60,0.699516
70,0.790539
80,0.764205
90,0.680541
100,0.727333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.6226677101192666, metrics={'train_runtime': 309.6364, 'train_samples_per_second': 15.444, 'train_steps_per_second': 1.931, 'total_flos': 629115478631424.0, 'train_loss': 0.6226677101192666, 'epoch': 2.0})

In [24]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.632231,0.616184,598


{'eval_loss': 0.6161841750144958}

In [25]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [26]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.71      0.78      0.74       329
           1       0.69      0.61      0.64       269

    accuracy                           0.70       598
   macro avg       0.70      0.69      0.69       598
weighted avg       0.70      0.70      0.70       598



In [27]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[255  74]
 [106 163]]


### DistilBert

In [28]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [29]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=256)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=256)

In [30]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [38]:
from transformers import AutoModelForSequenceClassification,Trainer,TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10
)

In [39]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english").to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Step,Training Loss
10,2.440433
20,1.809518
30,1.604457
40,1.117360
50,0.938839
60,0.654990
70,0.769967
80,0.795082
90,0.660366
100,0.729377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.6756964790382513, metrics={'train_runtime': 130.4778, 'train_samples_per_second': 36.65, 'train_steps_per_second': 4.583, 'total_flos': 316729550186496.0, 'train_loss': 0.6756964790382513, 'epoch': 2.0})

In [40]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.560855,0.619405,598


{'eval_loss': 0.6194052696228027}

In [41]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [42]:
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

           0       0.68      0.78      0.73       329
           1       0.67      0.56      0.61       269

    accuracy                           0.68       598
   macro avg       0.68      0.67      0.67       598
weighted avg       0.68      0.68      0.67       598



In [43]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[256  73]
 [119 150]]


### twitter-xlm-roberta

In [44]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [45]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [46]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)

In [53]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10
)

In [54]:
from transformers import AutoModelForSequenceClassification, Trainer


model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,0.999739
20,0.777220
30,0.740942
40,0.688612
50,0.703397
60,0.706232
70,0.752059
80,0.770604
90,0.700589
100,0.746059


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.6426707262976911, metrics={'train_runtime': 235.6791, 'train_samples_per_second': 20.29, 'train_steps_per_second': 2.537, 'total_flos': 314552090893824.0, 'train_loss': 0.6426707262976911, 'epoch': 2.0})

In [55]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Step
0.566380,0.626729,598


{'eval_loss': 0.6267291903495789}

In [56]:
y_pred = np.argmax(trainer.predict(test_dataset).predictions, axis=-1)

In [57]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.67      0.80      0.73       329
           1       0.68      0.52      0.59       269

    accuracy                           0.67       598
   macro avg       0.67      0.66      0.66       598
weighted avg       0.67      0.67      0.66       598



In [58]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[263  66]
 [130 139]]
